In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re
import time

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Base Models
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, 
    VotingClassifier, 
    StackingClassifier,
    ExtraTreesClassifier
)
from sklearn.naive_bayes import MultinomialNB, ComplementNB

# Advanced Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import RandomOverSampler

import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"\n📚 Key library versions:")
print(f"  - pandas: {pd.__version__}")
print(f"  - scikit-learn: {__import__('sklearn').__version__}")

In [ ]:
df = pd.read_excel("bharatfakenewskosh.xlsx")

statement_col = 'Eng_Trans_Statement'
body_col = 'Eng_Trans_News_Body'
label_col = 'Label'

if label_col not in df.columns:
    raise KeyError(f"Column '{label_col}' not found in dataset.")

# Keep all three columns
df = df[[statement_col, body_col, label_col]].dropna()

print(f"✅ Data loaded: {len(df)} samples")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nOriginal label distribution:")
print(df[label_col].value_counts())


In [ ]:
print("\n" + "="*80)
print("🔍 INITIAL DATA ANALYSIS")
print("="*80)

# Show sample statements to understand structure
print("\n📝 Sample Statements (showing fact-checking artifacts):")
for i in range(3):
    print(f"\n{i+1}. {df[statement_col].iloc[i][:150]}...")

# Check for fact-checking language
df_temp = df.copy()
df_temp['stmt_lower'] = df_temp[statement_col].astype(str).str.lower()

print("\n🔍 Detecting Fact-Checking Artifacts:")
artifacts = {
    'Starts with "Fact-check:"': df_temp['stmt_lower'].str.startswith('fact-check').sum(),
    'Contains "wrong claim"': df_temp['stmt_lower'].str.contains('wrong claim').sum(),
    'Contains "false claim"': df_temp['stmt_lower'].str.contains('false claim').sum(),
    'Contains "misleading"': df_temp['stmt_lower'].str.contains('misleading').sum(),
    'Contains "?"': df_temp['stmt_lower'].str.contains('\?', regex=True).sum(),
    'Contains "old video"': df_temp['stmt_lower'].str.contains('old video').sum(),
}

for artifact, count in artifacts.items():
    pct = (count / len(df)) * 100
    print(f"  {artifact}: {count} ({pct:.1f}%)")

print("\n⚠️  CRITICAL: These artifacts leak label information!")
print("   They must be removed for fair classification.")

In [ ]:
print("\n" + "="*80)
print("🔧 APPLYING AGGRESSIVE CLEANING")
print("="*80)

def clean_statement_aggressive(text):
    """
    Remove fact-checking artifacts that leak label information
    """
    text = str(text).lower()
    
    # Step 1: Remove everything before first colon (removes "Fact-check:", "Wrong claim:", etc.)
    if ':' in text:
        parts = text.split(':', 1)
        # Only remove if it's likely a fact-check prefix
        if any(word in parts[0] for word in ['fact', 'check', 'wrong', 'false', 'misleading', 'claim']):
            text = parts[1]
    
    # Step 2: Remove explicit fact-checking language
    leak_phrases = [
        'fact check', 'factcheck', 'fact-check',
        'wrong claim', 'false claim', 'misleading claim',
        'viral claim', 'fake news', 'hoax',
        'old video', 'old picture', 'old image', 'old photo',
        'share by stating', 'shared by stating',
        'edited', 'morphed', 'photoshopped',
        'out of context', 'misleading', 'unrelated',
        'media gave wrong', 'wrong news',
        'media shared wrong', 'shared wrong',
        'viral post', 'viral message',
        'debunked', 'busted'
    ]
    
    for phrase in leak_phrases:
        text = text.replace(phrase, ' ')
    
    # Step 3: Remove question marks and exclamation marks (signal uncertainty/sensationalism)
    text = text.replace('?', ' ').replace('!', ' ')
    
    # Step 4: Remove URLs
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+', '', text)
    
    # Step 5: Remove social media handles and hashtags
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    
    # Step 6: Remove numbers (dates, stats often used in fact-checks)
    text = re.sub(r'\d+', '', text)
    
    # Step 7: Keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Step 8: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("\n1️⃣ Cleaning statements (removing fact-checking artifacts)...")
df['statement_clean'] = df[statement_col].apply(clean_statement_aggressive)

# Show before/after examples
print("\n📋 BEFORE/AFTER Cleaning Examples:")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"BEFORE: {df[statement_col].iloc[i][:100]}...")
    print(f"AFTER:  {df['statement_clean'].iloc[i][:100]}...")

# Remove rows where cleaning resulted in very short text
df['text_length'] = df['statement_clean'].str.split().str.len()
before_filter = len(df)
df = df[df['text_length'] >= 5]  # Keep only statements with 5+ words
after_filter = len(df)
print(f"\n2️⃣ Filtered out {before_filter - after_filter} statements with <5 words")

# Standardize labels
print("\n3️⃣ Standardizing labels...")
df['Label_Original'] = df[label_col].astype(str).str.strip().str.upper()

# Map labels
if df['Label_Original'].str.contains('TRUE|FALSE').any():
    df['Label'] = df['Label_Original'].map({'TRUE': 1, 'FALSE': 0})
else:
    # Already numeric
    df['Label'] = df[label_col].astype(int)

# Remove any rows where mapping failed
df = df.dropna(subset=['Label'])
df['Label'] = df['Label'].astype(int)

print(f"\n4️⃣ Label distribution after cleaning:")
print(df['Label'].value_counts())
print(f"\nPercentages:")
print(df['Label'].value_counts(normalize=True))

# Remove duplicates based on cleaned statement
print("\n5️⃣ Removing duplicates...")
before_dup = len(df)
df = df.drop_duplicates(subset=['statement_clean'], keep='first')
after_dup = len(df)
print(f"Removed {before_dup - after_dup} duplicates")

print(f"\n✅ Final dataset size: {len(df)} samples")
print("="*80)


In [ ]:
print("\n" + "="*80)
print("📊 POST-CLEANING QUALITY CHECK")
print("="*80)

# Test separability after cleaning
def quick_test(texts, labels, name="Dataset"):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score
    
    vec = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
    X = vec.fit_transform(texts)
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X, labels, cv=3, scoring='accuracy')
    
    print(f"\n{name}:")
    print(f"  3-fold CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")
    print(f"  Individual folds: {scores}")
    
    if scores.mean() > 0.80:
        print(f"  ✅ EXCELLENT! High separability")
    elif scores.mean() > 0.70:
        print(f"  ✅ GOOD! Models should work well")
    elif scores.mean() > 0.60:
        print(f"  ⚠️  MODERATE: Challenging but workable")
    else:
        print(f"  ⚠️  LOW: Dataset is very difficult")
    
    return scores.mean()

cleaned_score = quick_test(df['statement_clean'], df['Label'], "Cleaned Statements")

print("\n🎯 Expected final model accuracy: {:.1f}% - {:.1f}%".format(
    cleaned_score * 100, 
    min(cleaned_score * 100 + 10, 95)
))


In [ ]:
print("\n" + "="*80)
print("📊 CREATING VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Class distribution
sns.countplot(x=df['Label'], ax=axes[0, 0], palette=['#FF6B6B', '#4ECDC4'])
axes[0, 0].set_title("Class Distribution\n(0=Fake, 1=True)", fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel("Label")
axes[0, 0].set_ylabel("Count")
for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container)

# 2. Text length distribution
df['word_count'] = df['statement_clean'].str.split().str.len()
sns.histplot(data=df, x='word_count', hue='Label', bins=30, ax=axes[0, 1], palette=['#FF6B6B', '#4ECDC4'])
axes[0, 1].set_title("Cleaned Statement Length Distribution", fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel("Word Count")

# 3. Boxplot by label
sns.boxplot(x='Label', y='word_count', data=df, ax=axes[1, 0], palette=['#FF6B6B', '#4ECDC4'])
axes[1, 0].set_title("Statement Length by Label", fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel("Label (0=Fake, 1=True)")
axes[1, 0].set_ylabel("Word Count")

# 4. Summary statistics
stats_text = f"""Dataset Summary:
━━━━━━━━━━━━━━━━━━━━━
Total Samples: {len(df):,}
Fake News: {(df['Label']==0).sum():,} ({(df['Label']==0).sum()/len(df)*100:.1f}%)
True News: {(df['Label']==1).sum():,} ({(df['Label']==1).sum()/len(df)*100:.1f}%)

Avg Words (Fake): {df[df['Label']==0]['word_count'].mean():.1f}
Avg Words (True): {df[df['Label']==1]['word_count'].mean():.1f}

Post-Cleaning CV Score: {cleaned_score:.3f}
Expected Model Range: {cleaned_score*100:.1f}%-{min(cleaned_score*100+10, 95):.1f}%
"""
axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
axes[1, 1].axis('off')
axes[1, 1].set_title("Summary Statistics", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('data_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: data_analysis.png")

# Word clouds
print("\n📊 Generating word clouds...")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

fake_text = " ".join(df[df['Label']==0]['statement_clean'].head(1000))
true_text = " ".join(df[df['Label']==1]['statement_clean'].head(1000))

if len(fake_text) > 0:
    wc_fake = WordCloud(width=700, height=400, background_color='black', 
                        colormap='Reds', max_words=100).generate(fake_text)
    axes[0].imshow(wc_fake)
    axes[0].set_title("Most Common Words - Fake News", fontsize=14, fontweight='bold')
    axes[0].axis("off")

if len(true_text) > 0:
    wc_true = WordCloud(width=700, height=400, background_color='black', 
                        colormap='Greens', max_words=100).generate(true_text)
    axes[1].imshow(wc_true)
    axes[1].set_title("Most Common Words - True News", fontsize=14, fontweight='bold')
    axes[1].axis("off")

plt.tight_layout()
plt.savefig('wordclouds_cleaned.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: wordclouds_cleaned.png")

In [ ]:
print("\n" + "="*80)
print("🔧 FEATURE EXTRACTION")
print("="*80)

# Optimized TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=7500,      # Sweet spot for this dataset size
    ngram_range=(1, 2),     # Unigrams + bigrams
    min_df=3,               # Ignore very rare terms
    max_df=0.85,            # Ignore very common terms
    sublinear_tf=True,      # Use log scaling
    strip_accents='unicode',
    lowercase=True,
    token_pattern=r'\b[a-zA-Z]{2,}\b'  # Words with 2+ letters only
)

# Use cleaned statements
X = df['statement_clean']
y = df['Label']

print(f"✓ Using cleaned statement column")
print(f"✓ Total samples: {len(X)}")

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\n✓ Train set: {len(X_train)} samples")
print(f"✓ Test set: {len(X_test)} samples")
print(f"✓ Train label distribution: {pd.Series(y_train).value_counts().to_dict()}")

# Create TF-IDF features
print(f"\n🔄 Creating TF-IDF features...")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"✓ Training feature shape: {X_train_tfidf.shape}")
print(f"✓ Test feature shape: {X_test_tfidf.shape}")
print(f"✓ Vocabulary size: {len(vectorizer.vocabulary_)}")

# Balanced oversampling (moderate, not aggressive)
print(f"\n🔄 Applying balanced oversampling...")
ros = RandomOverSampler(random_state=42, sampling_strategy=0.9)  # 90% balance
X_train_balanced, y_train_balanced = ros.fit_resample(X_train_tfidf, y_train)

print(f"✓ Balanced training shape: {X_train_balanced.shape}")
print(f"✓ Balanced label distribution: {pd.Series(y_train_balanced).value_counts().to_dict()}")

print("\n✅ Feature extraction complete!")


In [ ]:
print("\n" + "="*80)
print("🚀 TRAINING INDIVIDUAL MODELS")
print("="*80)

models = {
    # Linear Models (Fast & Effective for Text)
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1, random_state=42),
    "Ridge Classifier": RidgeClassifier(alpha=1.0, random_state=42),
    "SGD Classifier": SGDClassifier(loss='modified_huber', penalty='l2', alpha=0.0001, 
                                    max_iter=1000, n_jobs=-1, random_state=42),
    "Linear SVM": LinearSVC(C=1.0, max_iter=1000, dual=False, random_state=42),
    
    # Naive Bayes (Excellent for Text Classification)
    "Multinomial NB": MultinomialNB(alpha=0.1),
    "Complement NB": ComplementNB(alpha=0.1),
    
    # Tree-based Models
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=30, min_samples_split=10,
                                           n_jobs=-1, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=30, min_samples_split=10,
                                       n_jobs=-1, random_state=42),
    
    # Gradient Boosting (Optimized)
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
                            colsample_bytree=0.8, n_jobs=-1, random_state=42, 
                            eval_metric='logloss', tree_method='hist'),
    "LightGBM": LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
                              colsample_bytree=0.8, n_jobs=-1, random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1, 
                                  verbose=0, random_state=42, thread_count=-1)
}

results = {}
trained_models = {}
training_start = time.time()

for name, model in models.items():
    print(f"\n{'='*80}")
    print(f"🔄 Training: {name}")
    print(f"{'='*80}")
    
    start_time = time.time()
    
    try:
        model.fit(X_train_balanced, y_train_balanced)
        preds = model.predict(X_test_tfidf)
        
        train_time = time.time() - start_time
        
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        
        results[name] = {
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1': f1,
            'Time (s)': train_time
        }
        
        trained_models[name] = model
        
        print(f"✅ {name}")
        print(f"   Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
        print(f"   F1 Score:  {f1:.4f}")
        print(f"   Time:      {train_time:.1f}s")
        
    except Exception as e:
        print(f"❌ {name} failed: {str(e)}")

total_time = time.time() - training_start
print(f"\n✅ All individual models trained in {total_time/60:.1f} minutes!")

In [ ]:
# ==================== CELL 10: Train Ensemble Models ====================
print("\n" + "="*80)
print("🎯 TRAINING ENSEMBLE MODELS")
print("="*80)

# Ensemble 1: Soft Voting (Probability-based)
print("\n🔄 Training Soft Voting Ensemble...")
try:
    voting_soft = VotingClassifier(estimators=[
        ('lr', LogisticRegression(max_iter=1000, C=1.0, solver='saga', n_jobs=-1, random_state=42)),
        ('nb', ComplementNB(alpha=0.1)),
        ('xgb', XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, n_jobs=-1, 
                             random_state=42, eval_metric='logloss', tree_method='hist')),
        ('lgbm', LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, n_jobs=-1, 
                               random_state=42, verbose=-1))
    ], voting='soft', n_jobs=-1)
    
    voting_soft.fit(X_train_balanced, y_train_balanced)
    v_soft_preds = voting_soft.predict(X_test_tfidf)
    
    results["Voting Soft"] = {
        'Accuracy': accuracy_score(y_test, v_soft_preds),
        'Precision': precision_score(y_test, v_soft_preds),
        'Recall': recall_score(y_test, v_soft_preds),
        'F1': f1_score(y_test, v_soft_preds),
        'Time (s)': 0
    }
    trained_models["Voting Soft"] = voting_soft
    print(f"✅ Soft Voting: Acc={results['Voting Soft']['Accuracy']:.4f}")
except Exception as e:
    print(f"❌ Soft Voting failed: {str(e)}")

# Ensemble 2: Stacking (Multi-level)
print("\n🔄 Training Stacking Ensemble...")
try:
    stacking = StackingClassifier(estimators=[
        ('lr', LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)),
        ('nb', ComplementNB(alpha=0.1)),
        ('xgb', XGBClassifier(n_estimators=100, max_depth=6, n_jobs=-1, random_state=42, 
                             eval_metric='logloss', tree_method='hist')),
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=30, n_jobs=-1, random_state=42))
    ], final_estimator=LGBMClassifier(n_estimators=50, random_state=42, verbose=-1), 
    n_jobs=-1, cv=3)
    
    stacking.fit(X_train_balanced, y_train_balanced)
    s_preds = stacking.predict(X_test_tfidf)
    
    results["Stacking"] = {
        'Accuracy': accuracy_score(y_test, s_preds),
        'Precision': precision_score(y_test, s_preds),
        'Recall': recall_score(y_test, s_preds),
        'F1': f1_score(y_test, s_preds),
        'Time (s)': 0
    }
    trained_models["Stacking"] = stacking
    print(f"✅ Stacking: Acc={results['Stacking']['Accuracy']:.4f}")
except Exception as e:
    print(f"❌ Stacking failed: {str(e)}")

# Ensemble 3: Weighted Average
print("\n🔄 Creating Weighted Average Ensemble...")
try:
    # Get probabilities from best models
    model_probas = []
    weights = []
    
    if 'XGBoost' in trained_models:
        model_probas.append(trained_models['XGBoost'].predict_proba(X_test_tfidf)[:, 1])
        weights.append(0.30)
    if 'LightGBM' in trained_models:
        model_probas.append(trained_models['LightGBM'].predict_proba(X_test_tfidf)[:, 1])
        weights.append(0.30)
    if 'CatBoost' in trained_models:
        model_probas.append(trained_models['CatBoost'].predict_proba(X_test_tfidf)[:, 1])
        weights.append(0.25)
    if 'Complement NB' in trained_models:
        model_probas.append(trained_models['Complement NB'].predict_proba(X_test_tfidf)[:, 1])
        weights.append(0.15)
    
    if model_probas:
        # Normalize weights
        weights = np.array(weights) / sum(weights)
        weighted_proba = sum(w * p for w, p in zip(weights, model_probas))
        weighted_preds = (weighted_proba > 0.5).astype(int)
        
        results["Weighted Ensemble"] = {
            'Accuracy': accuracy_score(y_test, weighted_preds),
            'Precision': precision_score(y_test, weighted_preds),
            'Recall': recall_score(y_test, weighted_preds),
            'F1': f1_score(y_test, weighted_preds),
            'Time (s)': 0
        }
        print(f"✅ Weighted: Acc={results['Weighted Ensemble']['Accuracy']:.4f}")
except Exception as e:
    print(f"❌ Weighted Ensemble failed: {str(e)}")

print("\n✅ All ensemble models trained!")


In [ ]:
# ==================== CELL 11: Results Analysis ====================
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('F1', ascending=False)

print("\n" + "="*80)
print("📊 FINAL RESULTS - ALL MODELS RANKED BY F1 SCORE")
print("="*80)
print(results_df.to_string())

# Save results
results_df.to_csv('final_results.csv')
print("\n✅ Results saved to 'final_results.csv'")

# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 10 by Accuracy
top_10 = results_df.head(10)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_10)))

axes[0].barh(range(len(top_10)), top_10['Accuracy'], color=colors)
axes[0].set_yticks(range(len(top_10)))
axes[0].set_yticklabels(top_10.index)
axes[0].set_xlabel('Accuracy', fontsize=12)
axes[0].set_title('Top 10 Models - Test Accuracy', fontsize=14, fontweight='bold')
axes[0].axvline(x=0.80, color='red', linestyle='--', alpha=0.5, label='80% threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='x')

# Top 10 by F1
axes[1].barh(range(len(top_10)), top_10['F1'], color=plt.cm.plasma(np.linspace(0, 1, len(top_10))))
axes[1].set_yticks(range(len(top_10)))
axes[1].set_yticklabels(top_10.index)
axes[1].set_xlabel('F1 Score', fontsize=12)
axes[1].set_title('Top 10 Models - F1 Score', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: model_comparison.png")

# Performance metrics heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(results_df[['Accuracy', 'Precision', 'Recall', 'F1']].head(10), 
            annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'Score'},
            vmin=0.5, vmax=1.0)
plt.title("Top 10 Models - Performance Metrics Heatmap", fontsize=14, fontweight='bold')
plt.ylabel("Model", fontsize=12)
plt.xlabel("Metric", fontsize=12)
plt.tight_layout()
plt.savefig('performance_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: performance_heatmap.png")